# Bestand nach Nationalität, Aufenthaltsart und Geschlecht
bev_bestand_jahr_nationalitaet_aufenthaltsart_geschl_od3371

### Kurzbeschreibung
Bestand der wirtschaftlichen Wohnbevölkerung der Stadt Zürich nach Nationalität, Aufenthaltsart und Geschlecht.

Datum: 11.03.2026


Dataset auf PROD-Datakatalog: Link https://data.stadt-zuerich.ch/dataset/bev_bestand_jahr_nationalitaet_aufenthaltsart_geschl_od3371

Dataset auf INTEG-Datakatalog: Link https://data.integ.stadt-zuerich.ch/dataset/int_dwh_bev_bestand_jahr_nationalitaet_aufenthaltsart_geschl_od3371


### Importiere die notwendigen Packages

In [ ]:
#%pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime plotly leafmap

In [ ]:
import altair as alt
import datetime
import folium 
import geopandas as gpd
import io
from IPython.display import Markdown as md
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#import pivottablejs
#from pivottablejs import pivot_ui
import plotly.express as px
import requests
import seaborn as sns

Welche Python, Altair und Seaborn Version wird verwendet?

In [ ]:
#base env 2025: Python 3.11.7
import ipykernel
print(ipykernel.__version__)

import sys
import platform
print("Python-Version:", sys.version)
print("Python-Implementierung:", platform.python_implementation())
print("Python-Build:", platform.python_build())
print("Python-Compiler:", platform.python_compiler())

print("Altair-Version:", alt.__version__)
print("Seaborn-Version:", sns.__version__)

Importiere die eigenen Funktionen, die unter ../0_scripts abegelegt sind:

In [ ]:
import sys
sys.path.append('../0_scripts')

import my_py_dataviz_functions as mypy_dv
import my_py_dataloading_functions as mypy_dl

In [ ]:
SSL_VERIFY = False
# evtl. SSL_VERIFY auf False setzen wenn die Verbindung zu https://www.gemeinderat-zuerich.ch nicht klappt (z.B. wegen Proxy)
# Um die SSL Verifikation auszustellen, bitte die nächste Zeile einkommentieren ("#" entfernen)
# SSL_VERIFY = False

In [ ]:
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings()

### Settings
Definiere Settings. 
Hier das Zahlenformat von Float-Werten (z.B. *'{:,.2f}'.format* mit Komma als Tausenderzeichen)

In [ ]:
#pd.options.display.float_format = lambda x : '{:,.1f}'.format(x) if (np.isnan(x) | np.isinf(x)) else '{:,.0f}'.format(x) if int(x) == x else '{:,.1f}'.format(x)
pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 15)

#### Paletten aus Zuericolors
Die Farbwerte habe ich aus R ausgelesen. Siehe dazu: `G:\sszsim\myR\zuericolors4python`

In [ ]:
# Quantitative Paletten
zuericolors_qual12 = ["#3431DE", "#0A8DF6", "#23C3F1", "#7B4FB7", "#DB247D", "#FB737E", "#007C78", "#1F9E31", "#99C32E", "#9A5B01", "#FF720C", "#FBB900"]
zuericolors_qual12br = ["#5D4BFE", "#4AA9FF", "#55FFFF", "#986AD5", "#FC4C99", "#FF919A", "#349894", "#44B14A", "#B7E14E", "#B97624", "#FF7231", "#FFD736"]
zuericolors_qual12da= ["#0017BF", "#0072D7", "#00A5D2", "#5E359A", "#BA0062", "#DA5563", "#00615D", "#00770F", "#7BA600", "#7B4100", "#DC5500", "#DA9C00"]
# Divergente Paletten
zuericolors_div9val  =  ["#A30059", "#DB247D", "#FF579E", "#FFA8D0", "#E4E0DF", "#A8DBB1", "#55BC5D", "#1F9E31", "#10652A"] 
zuericolors_div9ntr  =  ["#782600", "#CC4309", "#FF720C", "#FFBC88", "#E4E0DF", "#AECBFF", "#6B8EFF", "#3B51FF", "#2F2ABB"] 
# Geschlechter Paletten
zuericolors_gender3  =  ["#349894", "#FFD736", "#986AD5"] 
zuericolors_gender6origin  =  ["#00615D", "#349894", "#DA9C00", "#FFD736", "#5E359A", "#986AD5"] 
zuericolors_gender5wedding  =  ["#349894", "#FFD736", "#3431DE", "#B8B8B8", "#D6D6D6"] 
# Sequenzielle Paletten
zuericolors_seq9blu  =  ["#CADEFF", "#AEC2FF", "#93A6FF", "#778AFF", "#5B6EFF", "#4D59E2", "#3E44C5", "#302FA7", "#211A8A"] 
zuericolors_seq9red  =  ["#FED2EE", "#FEAED6", "#F589BE", "#F165A5", "#ED408D", "#D1307B", "#B52069", "#991056", "#7D0044"] 
zuericolors_seq9grn  =  ["#CFEED8", "#A8E0B3", "#81D18F", "#5BC36A", "#34B446", "#2A9A3C", "#208032", "#166529", "#0C4B1F"] 
zuericolors_seq9brn  =  ["#FCDDBB", "#F7BD8C", "#F39D5E", "#EE7D2F", "#EA5D00", "#C84E00", "#A53E00", "#832F00", "#611F00"]

#### Zeitvariabeln


In [ ]:
#Zeitvariabeln als Strings:
now = datetime.date.today()
year_today = now.strftime("%Y")
date_today = "_"+now.strftime("%Y-%m-%d")

#Zeitvariabeln als Integers:
int_times = now.timetuple()
aktuellesJahr = int_times[0]
aktuellerMonat = int_times[1]
selectedMonat = int_times[1]-2
#print(aktuellesJahr, aktuellerMonat,'datenstand: ', selectedMonat, int_times)

In [ ]:
package_name = "bev_bestand_jahr_nationalitaet_aufenthaltsart_geschl_od3371"

In [ ]:
data2betested = mypy_dl.load_data(
    status = 'int'
    , data_source = 'web'
    , package_name = package_name
    , dataset_name = "BEV337OD3371"    
    , datums_attr = ['StichtagDatJahr']
    )

In [ ]:
data2betested.head(2).T

Berechne weitere Attribute falls notwendig

In [ ]:
data2betested = (
    data2betested
    .copy()
    .assign(
        StichtagDatJahr_str = lambda x: x.StichtagDatJahr.astype(str),
        Jahr = lambda x: x.StichtagDatJahr,
        Jahr_end = lambda x: x.StichtagDatJahr+pd.offsets.YearEnd(0),
        Jahr_nbr = lambda x: x.Jahr.dt.year,
    )
    .sort_values('StichtagDatJahr', ascending=False)
    )
data2betested.dtypes

Minimales und maximales Jahr im Datensatz

In [ ]:
data_max_jahr = str(max(data2betested.Jahr).year)
data_min_jahr = str(min(data2betested.Jahr).year)

print(f"Die Daten haben ein Minimumjahr von {data_min_jahr} und ein Maximumjahr von {data_max_jahr}")

In [ ]:
data_max_date = str(max(data2betested.Jahr).year)
data_min_date = str(min(data2betested.Jahr).year)

print(f"Die Daten haben ein Minimumjahr von {data_min_date} und ein Maximumjahr von {data_max_date}")

### Einfache Datentests

In [ ]:
data2betested.info(memory_usage='deep', verbose=True)

In [ ]:
print(f'The dataset has {data2betested.shape[0]:,.0f} rows (observations) and {data2betested.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {data2betested.duplicated().sum()} exact duplicates in the data.')

Beschreibe einzelne Attribute

In [ ]:
data2betested.describe()

Welches sind die Zeilen ohne Werte bei AnzBestWir?

In [ ]:
data2betested[np.isnan(data2betested.AnzBestWir)]

### Verwende das Datum als Index

While we did already parse the `datetime` column into the respective datetime type, it currently is just a regular column. 
**To enable quick and convenient queries and aggregations, we need to turn it into the index of the DataFrame**

In [ ]:
data2betested = data2betested.set_index("StichtagDatJahr")
data2betested = data2betested.sort_index()

In [ ]:
data2betested.index.year.unique()

### Beschreibe einzelne Attribute

Beschreibe nicht numerische Attribute

In [ ]:
# describe non-numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(data2betested.describe(exclude='number'))
except:
    print("No categorical data in dataset.")

Beschreibe numerische Attribute

In [ ]:
# describe numerical features
try:
    with pd.option_context('display.float_format', '{:,.0f}'.format):
        display(data2betested.describe(include='number'))
except:
    print("No numercial data in dataset.")

In [ ]:
plt.style.use('ggplot')
params = {'text.color': (0.25, 0.25, 0.25), 'figure.figsize': [18, 6]}
plt.rcParams.update(params)

try:
    data2betested.hist(bins=25, rwidth=0.9)
    plt.tight_layout()
    plt.show()
except:
    print("No numercial data to plot.")

### Gibt es Duplikate?

In [ ]:
# find duplicate rows
duplicate_rows = data2betested[data2betested.duplicated()]
duplicate_rows

### Nullwerte und Missings?

In [ ]:
data2betested.isnull().sum()

In [ ]:
# check missing values with missingno
#import missingno as msno
#msno.matrix(data2betested, labels=True, sort='descending');
#msno.heatmap(data2betested)

### Gruppierungen

In [ ]:
agg_jahr = data2betested.loc[data_min_date:data_max_date]\
    .groupby(['Jahr', 'Jahr_nbr', 'Jahr_end']) \
    .agg(sum_AnzBestWir=('AnzBestWir', 'sum')) \
    .sort_values('Jahr', ascending=False) 
agg_jahr.reset_index().head(3)

In [ ]:
agg_natlang = data2betested.loc[data_min_date:data_max_date]\
    .groupby(['NatLang', 'NatCd', 'AufenthaltsartLang', 'AufenthaltsartCd', 'SexLang', 'SexCd']) \
    .agg(sum_AnzBestWir=('AnzBestWir', 'sum')) \
    .sort_values('NatLang', ascending=True) 
agg_natlang.reset_index().head(5)

In [ ]:
agg_raum_zeit = data2betested.loc[data_min_date:data_max_date]\
    .groupby(['Jahr_nbr', 'Jahr_end', 'NatLang', 'NatCd', 'AufenthaltsartLang', 'AufenthaltsartCd', 'SexLang', 'SexCd']) \
    .agg(sum_AnzBestWir=('AnzBestWir', 'sum')) \
    .sort_values('Jahr_nbr', ascending=True) 
agg_raum_zeit.reset_index().head(3)

In [ ]:
data2betested.reset_index().columns

### Pivotiere

In [ ]:
pivoted_df = data2betested.pivot_table(
    index='Jahr_nbr',
    columns= ('NatLang', 'NatCd', 'AufenthaltsartLang', 'AufenthaltsartCd', 'SexLang', 'SexCd'),
    values='AnzBestWir',
    aggfunc='sum'
)

pivoted_df = pivoted_df.sort_index(ascending=False)

print(pivoted_df.head(8).T)

### Visualisierungen nach Zeitausschnitten

#### Entwicklung Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität

In [ ]:
myAgg1 = data2betested.loc[data_min_date:data_max_date]\
    .groupby(['StichtagDatJahr', 'NatLang', 'NatCd', 'AufenthaltsartLang', 'AufenthaltsartCd', 'SexLang', 'SexCd']) \
    .agg(sum_AnzBestWir=('AnzBestWir', 'sum')) \
    .sort_values('StichtagDatJahr', ascending=True) 

myAgg1.reset_index().head(3)

In [ ]:
grafik1 = mypy_dv.plot_altair_multiline_highlight(
    data = myAgg1.reset_index().sort_values('NatLang', ascending=True)
    ,x = 'StichtagDatJahr:T'
    ,y = 'sum_AnzBestWir:Q'
    ,x_beschriftung = 'Jahr'
    , y_beschriftung = 'Anz. Bestand'
    ,category = "NatLang:N"
    ,category_beschriftung= 'Legende:'
    ,x_sort = None
    ,palette_scheme = None
    ,custom_palette = zuericolors_qual12br
    ,line_width = 1.1
    ,warning_status = "ignore"
    ,myTitle="Entwicklung Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität, seit "+data_min_date
)
grafik1

##### Kombinierte Grafik in Altair

In [ ]:
top_row = alt.hconcat(grafik1, grafik1)
combined_chart = alt.vconcat(grafik1, grafik1, spacing=20)

final_chart = combined_chart.properties(
    title="Alle Teilgrafiken in einer zusammengesetzt:    ",
    background="#FDFDFD",
    padding={"left": 20, "top": 20, "right": 20, "bottom": 20},
    autosize={"type": "fit", "contains": "padding"}
)
final_chart = final_chart.resolve_scale(color='independent', shape='independent', size='independent')
final_chart

#### Barcharts mit Seaborn

In [ ]:
myAggBar = data2betested.loc[data_min_date:data_max_date]\
    .groupby(['StichtagDatJahr'] + ['NatLang', 'NatCd', 'AufenthaltsartLang', 'AufenthaltsartCd', 'SexLang', 'SexCd'] + ['Jahr', 'Jahr_end', 'Jahr_nbr']) \
    .agg(sum_AnzBestWir=('AnzBestWir', 'sum')) \
    .sort_values('StichtagDatJahr', ascending=True)

myAggBar.reset_index().head(3)

In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
myHist = sns.catplot(x="NatLang"
            , y="sum_AnzBestWir"
            , kind="bar"
            , palette="pastel"
            , height=5
            , aspect=3
            , order=None, legend_out=True
            ,data=myAggBar.reset_index().query(f"Jahr_nbr == {data_max_jahr}")
           )
myHist.set_xticklabels(rotation=45)
myHist.set_xlabels('Nationalität', fontsize=11)
myHist.set_ylabels('Anz. Bestand', fontsize=11)

##### Stacked Bar Chart

In [ ]:
data = myAggBar.query("sum_AnzBestWir > 0").reset_index()
data_pivoted = data.pivot(index='Jahr_nbr', columns='NatLang', values='sum_AnzBestWir').fillna(0)

colors = sns.color_palette("cubehelix", n_colors=len(data_pivoted.columns))
fig, ax = plt.subplots(figsize=(12, 6))
data_pivoted.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.9)
plt.title('Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität seit '+data_min_date, fontsize=14)
ax.set_xlabel('Jahr', fontsize=11)
ax.set_ylabel('Anz. Bestand', fontsize=11)
plt.legend(title='Nationalität', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Faced Grids

In [ ]:
data2betested.columns

In [ ]:
myFG = data2betested.reset_index()
myFG.head(3)

In [ ]:
faced_grid1 = mypy_dv.plot_sns_facetgrid(
    data = myFG.reset_index().sort_values('NatLang', ascending=True)
    ,col = "NatLang"
    ,hue = "NatLang"
    ,col_wrap = 5
    ,grafiktyp = sns.lineplot
    ,x = "StichtagDatJahr"
    ,y = "AnzBestWir"
    ,ylabel= "Anz. Bestand"
    ,warning_status ="ignore"
    ,height = 3
    ,myTitle="Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität, seit "+str(int(data2betested.index.year.min()))
)
faced_grid1

#### Treemaps

**Funktion zum einfärben**

Muss ich noch als Funktion umsetzen 

In [ ]:
attr2becolored = data2betested['NatLang'].unique().tolist()
verfügbare_farben_zuericolors = zuericolors_qual12da+zuericolors_qual12br+zuericolors_qual12+zuericolors_div9ntr

farben_dict_zc = {'(?)':'lightgrey'}
for index, x in enumerate(attr2becolored):
    farben_dict_zc[x] = verfügbare_farben_zuericolors[index % len(verfügbare_farben_zuericolors)]

print(farben_dict_zc)

In [ ]:
data2betested.columns

Jahre definieren, die dargestellt werden sollen

In [ ]:
int_data_max_year = data2betested.index.max().year

years = [
    int_data_max_year - 20,
    int_data_max_year
]
print(years)

##### Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität und Jahr

In [ ]:
myTM = data2betested.loc[data_min_date:data_max_date].reset_index()

myTM.reset_index().head(2)

In [ ]:
treeMap1 = mypy_dv.plot_px_treemap(
    data=myTM.reset_index().query("AnzBestWir>0")
    ,levels=['NatLang', 'Jahr_nbr']
    ,values="AnzBestWir"
    ,color="AnzBestWir"
    ,color_discrete_map={'(?)':'lightgrey'}
    ,height=600
    ,width=1100
    ,myHeaderTitle="Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität und Jahr, seit "+data_min_date
)
treeMap1

##### Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität (aktuellstes Jahr)

In [ ]:
myTM2 = data2betested.loc[data_max_date]

myTM2.reset_index().head(2)

In [ ]:
treeMap2 = mypy_dv.plot_px_treemap(
    data=myTM2.reset_index().query("AnzBestWir>0")
    ,levels=['NatLang']
    ,values="AnzBestWir"
    ,color="NatLang"
    ,color_discrete_map=farben_dict_zc
    ,height=600
    ,width=1100
    ,myHeaderTitle="Bestand nach Nationalität, Aufenthaltsart und Geschlecht nach Nationalität am "+data_max_date
)
treeMap2

## ---------------------- hier Plausi beendet

**Sharepoint als gecheckt markieren!**

Record auf Sharepoint: **[Link]()**

---------------------------------------------------------------------------